# 01b — Synthesize EN→puhekieli pairs (back-translation)

Rap lyrics are the register we want but are **FI-only**. OpenSubtitles gives real
EN→FI pairs, but its Finnish is only *mildly* colloquial — so a model trained on it
drifts toward safe, semi-formal Finnish. To teach it to **produce hard rap-register
puhekieli from English**, we manufacture the missing English side:

```
real FI lyric  --(local LLM: FI→EN)-->  synthetic EN
training pair  =  (synthetic EN  →  real FI lyric)
```

The **Finnish side is always the authentic lyric**, so the target the model learns
to generate is genuine puhekieli. Only the English input is synthetic (input noise
matters far less than output noise). This is standard low-resource MT practice.

Runs against **LM Studio** (OpenAI-compatible). The endpoint/model come from your
`.env` (`LMSTUDIO_BASE_URL`, `LMSTUDIO_MODEL`). Nothing leaves the machine. Load a
model in LM Studio and start its server first.

In [1]:
from puhekieli_llm.config import LMSTUDIO_BASE_URL, LMSTUDIO_MODEL, CLEAN
from puhekieli_llm import synth
print('endpoint:', LMSTUDIO_BASE_URL)
print('model   :', LMSTUDIO_MODEL, '(override via LMSTUDIO_MODEL env)')
# quick connectivity + quality probe on one line
try:
    demo = synth.back_translate_line('mä oon stadin kundi ja me mennään keikalle')
    print('FI->EN probe:', demo)
except Exception as e:
    print('LM Studio not reachable:', e)
    print('Start LM Studio, load a model, enable the local server, then re-run.')

endpoint: http://172.20.10.7:1234/v1
model   : gpt-oss-20b (override via LMSTUDIO_MODEL env)
FI->EN probe: I’m a city guy and we’re heading out for a gig.


## Generate pairs

Resumable: skips ids already in `rap_synthetic.jsonl` and flushes every 25 lines,
so a crash/interrupt won't lose progress. Start with a small `limit` to eyeball
quality before running the whole set.

In [5]:
# start small; raise once the output looks good
if (CLEAN / 'genius_rap.jsonl').exists():
    try:
        synth.synthesize_pairs(limit=6000)
    except Exception as e:
        print('back-translation skipped (LM Studio not reachable?):', e)
else:
    print('No genius_rap.jsonl yet — run 01_collect.ipynb (fetch + clean) first.')

resuming: 949 pairs already synthesized
  1025/6000 back-translated
  1075/6000 back-translated
  1100/6000 back-translated
  1150/6000 back-translated
  1175/6000 back-translated
  1200/6000 back-translated
  1225/6000 back-translated
  1250/6000 back-translated
  1275/6000 back-translated
  1300/6000 back-translated
  1325/6000 back-translated
  1350/6000 back-translated
  1375/6000 back-translated
  1400/6000 back-translated
  1425/6000 back-translated
  1450/6000 back-translated
  1500/6000 back-translated
  1525/6000 back-translated
  1550/6000 back-translated
  1575/6000 back-translated
  1600/6000 back-translated
  1625/6000 back-translated
  1650/6000 back-translated
  1675/6000 back-translated
  1700/6000 back-translated
  1725/6000 back-translated
  1750/6000 back-translated
  1800/6000 back-translated
  1825/6000 back-translated
  1850/6000 back-translated
  1875/6000 back-translated
  1900/6000 back-translated
  1925/6000 back-translated
  1950/6000 back-translated
  2000/6

In [6]:
# eyeball the synthetic pairs + confirm FI side stays puhekieli
from puhekieli_llm.sources import read_jsonl
from puhekieli_llm.eval import puhekieli_score, corpus_puhekieli_rate
p = CLEAN / 'rap_synthetic.jsonl'
if p.exists():
    recs = list(read_jsonl(p))
    print(f'{len(recs)} synthetic pairs\n')
    for r in recs[:8]:
        print(f"  EN (synth): {r['en']}")
        print(f"  FI (real) : [{puhekieli_score(r['fi'])['score']:+.1f}] {r['fi']}\n")
    print(f"FI side spoken-leaning fraction: {corpus_puhekieli_rate([r['fi'] for r in recs]):.0%}")
else:
    print('No synthetic pairs yet — run the cell above.')

5480 synthetic pairs

  EN (synth): I can take you around the world
  FI (real) : [+1.0] Mä voin viedä maailman ympäri sut

  EN (synth): Whenever I'm with you, there's always a birthday, boo.
  FI (real) : [+1.0] Mun kaa sulla on aina synttärit, boo

  EN (synth): Why are you thinking about them? We could head over to Bogotá.
  FI (real) : [+1.0] Miks sä mietit niitä? Voidaan lähtee Bogotaan

  EN (synth): You're not coming over today—just staying at home again.
  FI (real) : [+1.0] Tätä päivää sä et tuu viettämään ny jälleen kotona

  EN (synth): I can get you out of the office world.
  FI (real) : [+1.0] Mä voin viedä maailmal konttorist sut (Ooh-wee)

  EN (synth): I can take you around the world (Okay)
  FI (real) : [+1.0] Mä voin viedä maailman ympäri sut (Okay)

  EN (synth): Whenever I’m with you it’s always your birthday, boo.
  FI (real) : [+1.0] Mun kaa sul on aina synttärit, boo

  EN (synth): And all those feelings surprised me (Action, action, action)
  FI (real) : [+1.0]

## ✅ Next
Once the pairs look reasonable, raise `limit` (e.g. to all lyrics), let it run,
then head to `02_tokenizer.ipynb`. Training data will be:
`opensubtitles_enfi` (base ability) + `rap_synthetic` (puhekieli push).